# Consistency Analysis: Lap-to-Lap Variation

This notebook analyzes your driving consistency by measuring variation in braking points, corner speeds, and throttle application across all laps.

## What You'll Find Here

- **Braking Point Consistency**: Box plots showing variation in where you start braking for each corner
- **Corner Speed Consistency**: Box plots showing minimum speed variation through each corner
- **Exit Speed Consistency**: Box plots showing corner exit speed variation — inconsistent exits cost time on the following straight
- **Exit Speed Opportunity Analysis**: Ranks corners by how much exit speed inconsistency matters, factoring in the length of the acceleration zone that follows
- **Throttle Acceptance**: The lateral G at which you reach full throttle during corner exit, as a percentage of the corner's peak lateral G
- **Summary Statistics Table**: Mean, standard deviation, min, max, and range for each segment

## How to Interpret the Results

- **Tight box plots** (small range): Consistent performance - you're hitting the same marks each lap
- **Wide box plots** (large range): Inconsistent - opportunity for improvement through practice
- **Outliers** (dots outside whiskers): Unusual laps - could be mistakes, traffic, or experimenting with lines
- **High standard deviation**: Focus area for practice
- **High throttle acceptance %**: Getting on full throttle earlier while still cornering hard - more aggressive exit

## Using Your Own Data

To analyze your own data:

1. **Run the first cell** below to install packages and display the upload widget
2. **Click "Choose File"** to select your `.xrk`, `.xrz`, or `.ibt` file
3. **Run all remaining cells** to analyze your data

The status indicator will show which file is being used. If you don't upload a file, the sample data will be used.

The notebook will automatically:
- Detect corners and zones from your GPS and pedal data
- Analyze all valid laps (excluding pit laps)
- Generate consistency metrics for each track segment

## Requirements

- GPS data channels (`GPS Latitude`, `GPS Longitude`, `GPS Speed`)
- Brake pressure (`BrakePress`) and throttle (`PPS`)
- Lateral acceleration (`LateralAcc`) for throttle acceptance analysis
- Multiple laps of data for meaningful consistency analysis

**Note:** This notebook works in both JupyterLite (browser) and standard JupyterLab environments.

In [ ]:
# Install required packages (needed for JupyterLite, skipped in regular JupyterLab if already installed)
%pip install -q motorsports-data-notebook

# Use the Rust parser backend for ~3x faster file loading
import os

os.environ["LIBXRK_BACKEND"] = "rust"

# Import core libraries
import pandas as pd
from IPython.display import display

# Visualization libraries
import plotly.express as px
import plotly.graph_objects as go

# Import helper functions
from motorsports_data_notebook.channels import (
    get_best_lap_channels,
    get_top_laps,
)
from motorsports_data_notebook.corners import identify_corners
from motorsports_data_notebook.driver_analysis import find_throttle_acceptance
from motorsports_data_notebook.visualization import (
    format_lap_time,
    plot_track_segments,
    visualize_throttle_acceptance,
    show_fig,
)
from motorsports_data_notebook.widgets import SessionPicker
from motorsports_data_notebook.zones import (
    compute_segment_stats,
    create_track_segments,
    detect_zones_averaged,
    get_corner_data,
)

# Session picker with channel configuration
# Upload your own .xrk/.xrz/.ibt file or use the sample data
session = SessionPicker(
    default_file="../data/CMD_Inferno 86_Fuji GP Sh_Generic testing_a_2248.xrz",
    channel_mapping={
        # GPS channels (required for corner detection)
        "gps_latitude": "GPS Latitude",
        "gps_longitude": "GPS Longitude",
        "gps_speed": "GPS Speed",  # Speed in m/s from GPS
        # Pedal inputs (required for zone detection)
        "throttle": "PPS",  # Throttle position sensor (0-100%)
        "brake": "BrakePress",  # Brake pressure (0-100%)
        # Dynamics (required for throttle acceptance analysis)
        "lateral_g": "LateralAcc",  # Lateral acceleration in G
        "steering": "SteerAngle",  # Steering angle in degrees
    },
)
session.display()

In [ ]:
# Get the loaded session data
log = session.get_log()
laps = session.get_laps()
CHANNEL_NAMES = session.get_channel_names()

In [ ]:
# Display lap times table
laps.style.format({"lap_time": format_lap_time})  # type: ignore[dict-item]

In [ ]:
# Extract best lap channel data using libxrk 0.5.0 methods
best_lap, channels = get_best_lap_channels(
    log, laps, [CHANNEL_NAMES["gps_latitude"], CHANNEL_NAMES["gps_longitude"], "distance_m"]
)

# Filter to best lap and resample to GPS timebase for corner detection
best_lap_num = int(best_lap["num"])
gps_lat_ch = CHANNEL_NAMES["gps_latitude"]
gps_lon_ch = CHANNEL_NAMES["gps_longitude"]
aligned = (
    log.filter_by_lap(best_lap_num)
    .select_channels([gps_lat_ch, gps_lon_ch, "distance_m"])
    .resample_to_channel(gps_lat_ch)
    .channels
)

# Convert to arrays for corner detection
lap_channels = {
    "GPS Latitude": aligned[gps_lat_ch].column(gps_lat_ch).to_numpy(),
    "GPS Longitude": aligned[gps_lon_ch].column(gps_lon_ch).to_numpy(),
    "distance_m": aligned["distance_m"].column("distance_m").to_numpy(),
}

In [ ]:
# Identify corners directly from GPS coordinates
corners = identify_corners(
    lat=lap_channels["GPS Latitude"],
    lon=lap_channels["GPS Longitude"],
    threshold=0.003,  # Tuned on Fuji and Sodegaura
    min_corner_length=15,
    min_gap=80,
)

print(f"Found {len(corners)} corners")

In [ ]:
# Get top laps and detect zones averaged across them
top_laps = get_top_laps(laps, threshold_pct=1.03)

# Detect and average braking/acceleration zones across top laps
braking_zones, accel_zones = detect_zones_averaged(log, top_laps, CHANNEL_NAMES)

print(f"Found {len(braking_zones)} braking zones and {len(accel_zones)} acceleration zones")

In [ ]:
# Create track segments
track_length = lap_channels["distance_m"][-1]
segments = create_track_segments(corners, braking_zones, accel_zones, track_length)

print(f"Created {len(segments)} track segments")

In [ ]:
# Track map with labeled corners for reference
lap_channels_df = pd.DataFrame(lap_channels)
fig = plot_track_segments(lap_channels_df, segments, title="Track Map with Corners")
show_fig(fig)

In [ ]:
# Compute per-lap segment statistics
stats_df = compute_segment_stats(log, top_laps, segments, CHANNEL_NAMES)

print(f"Analyzing {len(top_laps)} laps (within 103% of best time)...")
print(f"Computed {len(stats_df)} segment statistics across all laps")

In [ ]:
# Visualize braking consistency
# Show braking point variation for each corner, centered around the mean

braking_stats = stats_df[stats_df["segment_type"] == "braking"].dropna(subset=["braking_point"])

if len(braking_stats) > 0:
    # Calculate deviation from mean braking point for each segment
    braking_stats = braking_stats.copy()
    braking_stats["braking_deviation"] = braking_stats.groupby("segment_name")[
        "braking_point"
    ].transform(lambda x: x - x.mean())

    fig = px.box(
        braking_stats,
        x="segment_name",
        y="braking_deviation",
        title="Braking Point Consistency by Corner (Centered on Mean)",
        labels={"braking_deviation": "Deviation from Mean (m)", "segment_name": "Corner"},
    )
    fig.update_layout(xaxis_tickangle=-45, width=900, height=500)
    # Add a reference line at zero (the mean)
    fig.add_hline(y=0, line_dash="dash", line_color="gray", opacity=0.5)
    show_fig(fig)
else:
    print("No braking data available")

In [ ]:
# Visualize corner minimum speed consistency
corner_stats = stats_df[stats_df["segment_type"] == "corner"].dropna(subset=["min_speed"])

if len(corner_stats) > 0:
    fig = px.box(
        corner_stats,
        x="segment_name",
        y="min_speed",
        title="Minimum Corner Speed Consistency",
        labels={"min_speed": "Min Speed (km/h)", "segment_name": "Corner"},
    )
    fig.update_layout(xaxis_tickangle=-45, width=900, height=500)
    show_fig(fig)
else:
    print("No corner speed data available")

In [ ]:
# Visualize corner exit speed consistency
exit_speed_stats = stats_df[stats_df["segment_type"] == "corner"].dropna(subset=["exit_speed"])

if len(exit_speed_stats) > 0:
    fig = px.box(
        exit_speed_stats,
        x="segment_name",
        y="exit_speed",
        title="Corner Exit Speed Consistency",
        labels={"exit_speed": "Exit Speed (km/h)", "segment_name": "Corner"},
    )
    fig.update_layout(xaxis_tickangle=-45, width=900, height=500)
    show_fig(fig)
else:
    print("No exit speed data available")

In [ ]:
# Exit speed opportunity analysis
# Ranks corners by exit_speed_std × accel_zone_length — higher = more time on the table

if len(exit_speed_stats) > 0:
    # Compute per-corner exit speed statistics
    exit_grouped = (
        exit_speed_stats.groupby(["segment_name", "corner_id"])["exit_speed"]
        .agg(["std", "mean"])
        .reset_index()
    )
    exit_grouped.columns = ["segment_name", "corner_id", "exit_speed_std", "exit_speed_mean"]

    # Map each corner to its acceleration zone length
    accel_segments = {
        seg.corner_id: seg.end_dist - seg.start_dist
        for seg in segments
        if seg.segment_type == "acceleration" and seg.corner_id is not None
    }
    exit_grouped["accel_zone_length_m"] = exit_grouped["corner_id"].map(accel_segments)
    exit_grouped = exit_grouped.dropna(subset=["accel_zone_length_m", "exit_speed_std"])

    # Compute opportunity score
    exit_grouped["opportunity_score"] = (
        exit_grouped["exit_speed_std"] * exit_grouped["accel_zone_length_m"]
    )
    exit_grouped = exit_grouped.sort_values("opportunity_score", ascending=False)

    print("Exit Speed Opportunity Analysis")
    print("(Higher score = more lap time left on the table)\n")
    display(
        exit_grouped[
            [
                "segment_name",
                "exit_speed_mean",
                "exit_speed_std",
                "accel_zone_length_m",
                "opportunity_score",
            ]
        ]
        .rename(
            columns={
                "segment_name": "Corner",
                "exit_speed_mean": "Mean Exit Speed (km/h)",
                "exit_speed_std": "Exit Speed Std (km/h)",
                "accel_zone_length_m": "Accel Zone Length (m)",
                "opportunity_score": "Opportunity Score",
            }
        )
        .reset_index(drop=True)
        .style.format(
            {
                "Mean Exit Speed (km/h)": "{:.1f}",
                "Exit Speed Std (km/h)": "{:.2f}",
                "Accel Zone Length (m)": "{:.0f}",
                "Opportunity Score": "{:.1f}",
            }
        )
    )
else:
    print("No exit speed data available for opportunity analysis")

In [ ]:
# Exit speed opportunity bar chart
if len(exit_speed_stats) > 0 and len(exit_grouped) > 0:
    plot_data = exit_grouped.sort_values("opportunity_score", ascending=True)

    fig = go.Figure()
    fig.add_trace(
        go.Bar(
            y=plot_data["segment_name"],
            x=plot_data["opportunity_score"],
            orientation="h",
            text=[
                f"\u03c3={std:.1f} km/h, {length:.0f}m"
                for std, length in zip(
                    plot_data["exit_speed_std"], plot_data["accel_zone_length_m"]
                )
            ],
            textposition="auto",
            hovertext=[
                f"{name}<br>Exit Speed Std: {std:.2f} km/h<br>"
                f"Accel Zone: {length:.0f}m<br>Score: {score:.1f}"
                for name, std, length, score in zip(
                    plot_data["segment_name"],
                    plot_data["exit_speed_std"],
                    plot_data["accel_zone_length_m"],
                    plot_data["opportunity_score"],
                )
            ],
            hoverinfo="text",
        )
    )
    fig.update_layout(
        title="Exit Speed Opportunity (Higher = More Time on the Table)",
        xaxis_title="Opportunity Score (Exit Speed Std × Accel Zone Length)",
        yaxis_title="Corner",
        width=900,
        height=max(400, len(plot_data) * 50 + 100),
    )
    show_fig(fig)

In [ ]:
# Compute throttle acceptance for each corner across all laps
# Throttle acceptance = lateral G at sustained full throttle / peak lateral G of corner

# Smoothing window for lateral G calculations and visualization
LATERAL_G_SMOOTHING_WINDOW = 1

# Channels needed for throttle acceptance analysis
throttle_ch = CHANNEL_NAMES["throttle"]
lateral_g_ch = CHANNEL_NAMES["lateral_g"]
THROTTLE_ACCEPTANCE_CHANNELS = ["distance_m", throttle_ch, lateral_g_ch]

throttle_acceptance_stats = []

for idx, lap in top_laps.iterrows():
    lap_num = int(lap["num"])

    # Use libxrk 0.5.0 methods to filter by lap, select channels, and resample
    aligned = (
        log.filter_by_lap(lap_num)
        .select_channels(THROTTLE_ACCEPTANCE_CHANNELS)
        .resample_to_channel("distance_m")
        .channels
    )

    # Build DataFrame with channels + timecodes (timecodes come from aligned tables)
    lap_data = pd.DataFrame(
        {name: aligned[name].column(name).to_numpy() for name in THROTTLE_ACCEPTANCE_CHANNELS}
    )
    # Add timecodes from reference channel's table
    lap_data["timecodes"] = aligned["distance_m"].column("timecodes").to_numpy()

    if len(lap_data) < 10:
        continue

    for corner in corners:
        result = find_throttle_acceptance(
            lap_data,
            corner,
            CHANNEL_NAMES,
            smoothing_window=LATERAL_G_SMOOTHING_WINDOW,
        )
        if result is not None:
            throttle_acceptance_stats.append(
                {
                    "corner_name": corner.name,
                    "corner_id": corner.id,
                    "lap_num": lap["num"],
                    "throttle_acceptance_pct": result["throttle_acceptance_pct"],
                    "lateral_g_at_throttle": result["lateral_g_at_throttle"],
                    "peak_lateral_g": result["peak_lateral_g"],
                    "full_throttle_dist": result["full_throttle_dist"],
                }
            )

throttle_acceptance_df = pd.DataFrame(throttle_acceptance_stats)
print(f"Computed throttle acceptance for {len(throttle_acceptance_df)} corner/lap combinations")

In [ ]:
# Visualize throttle acceptance consistency
# Shows at what percentage of peak lateral G the driver reaches full throttle

if len(throttle_acceptance_df) > 0:
    corner_order = [c.name for c in corners]
    fig = px.box(
        throttle_acceptance_df,
        x="corner_name",
        y="throttle_acceptance_pct",
        title="Throttle Acceptance by Corner (% of Peak Lateral G at Full Throttle)",
        labels={
            "throttle_acceptance_pct": "Throttle Acceptance (%)",
            "corner_name": "Corner",
        },
        category_orders={"corner_name": corner_order},
    )
    fig.update_layout(xaxis_tickangle=-45, width=900, height=500)
    # Add reference lines
    fig.add_hline(
        y=100,
        line_dash="dash",
        line_color="red",
        opacity=0.5,
        annotation_text="100% = Full throttle at peak G",
    )
    show_fig(fig)
else:
    print("No throttle acceptance data available")

In [ ]:
# Throttle acceptance summary statistics table
if len(throttle_acceptance_df) > 0:
    throttle_summary = []
    for corner_name in throttle_acceptance_df["corner_name"].unique():
        corner_data = throttle_acceptance_df[throttle_acceptance_df["corner_name"] == corner_name]
        throttle_summary.append(
            {
                "Corner": corner_name,
                "Mean (%)": corner_data["throttle_acceptance_pct"].mean(),
                "Std (%)": corner_data["throttle_acceptance_pct"].std(),
                "Min (%)": corner_data["throttle_acceptance_pct"].min(),
                "Max (%)": corner_data["throttle_acceptance_pct"].max(),
                "Range (%)": corner_data["throttle_acceptance_pct"].max()
                - corner_data["throttle_acceptance_pct"].min(),
                "N": len(corner_data),
            }
        )

    throttle_summary_df = pd.DataFrame(throttle_summary)
    display(
        throttle_summary_df.style.format(
            {
                "Mean (%)": "{:.1f}",
                "Std (%)": "{:.1f}",
                "Min (%)": "{:.1f}",
                "Max (%)": "{:.1f}",
                "Range (%)": "{:.1f}",
            }
        )
    )
else:
    print("No throttle acceptance data available")

In [ ]:
# Visualize throttle acceptance concept for Turn 1 (best lap)
# Shows throttle, brake, steering, and lateral G over distance with reference lines

turn1 = corners[0]

# Use libxrk 0.5.0 methods to get channels for best lap
best_lap_num = int(best_lap["num"])
best_lap_aligned = (
    log.filter_by_lap(best_lap_num)
    .select_channels(THROTTLE_ACCEPTANCE_CHANNELS)
    .resample_to_channel("distance_m")
    .channels
)
best_lap_df = pd.DataFrame(
    {name: best_lap_aligned[name].column(name).to_numpy() for name in THROTTLE_ACCEPTANCE_CHANNELS}
)
# Add timecodes from reference channel's table
best_lap_df["timecodes"] = best_lap_aligned["distance_m"].column("timecodes").to_numpy()

turn1_result = find_throttle_acceptance(
    best_lap_df, turn1, CHANNEL_NAMES, smoothing_window=LATERAL_G_SMOOTHING_WINDOW
)
assert turn1_result is not None, f"Could not compute throttle acceptance for {turn1.name}"

# Get corner data for best lap (receives pre-filtered log)
brake_ch = CHANNEL_NAMES["brake"]
steering_ch = CHANNEL_NAMES["steering"]
CORNER_VIZ_CHANNELS = ["distance_m", throttle_ch, brake_ch, lateral_g_ch, steering_ch]
lap_log = log.filter_by_lap(best_lap_num)
corner_data = get_corner_data(lap_log, turn1, CORNER_VIZ_CHANNELS, margin=50)

# Compute smoothed lateral G
lateral_g_smooth = (
    corner_data[lateral_g_ch]
    .abs()
    .rolling(window=LATERAL_G_SMOOTHING_WINDOW, center=True, min_periods=1)
    .mean()
)

fig = visualize_throttle_acceptance(
    distance=corner_data["distance_m"],
    throttle=corner_data[throttle_ch],
    lateral_g=lateral_g_smooth,
    corner=turn1,
    throttle_acceptance_result=turn1_result,
    brake=corner_data.get(brake_ch),
    steering=corner_data.get(steering_ch),
)
show_fig(fig)

In [ ]:
# Summary statistics table
def compute_summary_stats(stats_df):
    """Compute summary statistics for each segment across all laps."""
    summary = []

    # Braking segments
    for seg_name in stats_df[stats_df["segment_type"] == "braking"]["segment_name"].unique():
        seg_data = stats_df[
            (stats_df["segment_name"] == seg_name) & stats_df["braking_point"].notna()
        ]
        if len(seg_data) > 0:
            summary.append(
                {
                    "Segment": seg_name,
                    "Type": "Braking",
                    "Metric": "Braking Point (m)",
                    "Mean": seg_data["braking_point"].mean(),
                    "Std": seg_data["braking_point"].std(),
                    "Min": seg_data["braking_point"].min(),
                    "Max": seg_data["braking_point"].max(),
                    "Range": seg_data["braking_point"].max() - seg_data["braking_point"].min(),
                    "N": len(seg_data),
                }
            )

    # Corner segments
    for seg_name in stats_df[stats_df["segment_type"] == "corner"]["segment_name"].unique():
        seg_data = stats_df[(stats_df["segment_name"] == seg_name) & stats_df["min_speed"].notna()]
        if len(seg_data) > 0:
            summary.append(
                {
                    "Segment": seg_name,
                    "Type": "Corner",
                    "Metric": "Min Speed (km/h)",
                    "Mean": seg_data["min_speed"].mean(),
                    "Std": seg_data["min_speed"].std(),
                    "Min": seg_data["min_speed"].min(),
                    "Max": seg_data["min_speed"].max(),
                    "Range": seg_data["min_speed"].max() - seg_data["min_speed"].min(),
                    "N": len(seg_data),
                }
            )

        exit_data = stats_df[
            (stats_df["segment_name"] == seg_name) & stats_df["exit_speed"].notna()
        ]
        if len(exit_data) > 0:
            summary.append(
                {
                    "Segment": seg_name,
                    "Type": "Corner",
                    "Metric": "Exit Speed (km/h)",
                    "Mean": exit_data["exit_speed"].mean(),
                    "Std": exit_data["exit_speed"].std(),
                    "Min": exit_data["exit_speed"].min(),
                    "Max": exit_data["exit_speed"].max(),
                    "Range": exit_data["exit_speed"].max() - exit_data["exit_speed"].min(),
                    "N": len(exit_data),
                }
            )

    return pd.DataFrame(summary)


summary_df = compute_summary_stats(stats_df)
summary_df.style.format(
    {"Mean": "{:.1f}", "Std": "{:.1f}", "Min": "{:.1f}", "Max": "{:.1f}", "Range": "{:.1f}"}
)